# 04 - Seguridad, Managed Identity y Grants

Objetivo: centralizar la configuración administrativa de seguridad para el acceso a ADLS Gen2 mediante Managed Identity y Unity Catalog.

Este notebook contiene:

- Storage Credential para Managed Identity
- External Location hacia el contenedor RAW
- Grants sobre External Location, catálogo y esquemas

In [ ]:
# Widgets de configuración
# Los valores por defecto están alineados con databricks.yml.
# Reemplazar placeholders por valores reales antes de activar execute_admin_setup = true.

dbutils.widgets.text("catalog_name", "fintech_lakehouse", "Unity Catalog")
dbutils.widgets.text("raw_base_path", "abfss://raw@<storage-account>.dfs.core.windows.net/fintech", "ADLS Raw Base Path")
dbutils.widgets.text("storage_credential_name", "fintech_mi_credential", "Storage Credential")
dbutils.widgets.text("external_location_name", "fintech_raw_location", "External Location")
dbutils.widgets.text("access_connector_resource_id", "<access-connector-resource-id>", "Azure Databricks Access Connector Resource ID")
dbutils.widgets.text("security_group", "data_engineers", "Workspace group for data engineering")
dbutils.widgets.dropdown("execute_admin_setup", "false", ["false", "true"], "Execute admin security setup")

catalog_name = dbutils.widgets.get("catalog_name")
raw_base_path = dbutils.widgets.get("raw_base_path")
storage_credential_name = dbutils.widgets.get("storage_credential_name")
external_location_name = dbutils.widgets.get("external_location_name")
access_connector_resource_id = dbutils.widgets.get("access_connector_resource_id")
security_group = dbutils.widgets.get("security_group")
execute_admin_setup = dbutils.widgets.get("execute_admin_setup").lower() == "true"

print(f"Catalog: {catalog_name}")
print(f"Raw base path: {raw_base_path}")
print(f"Storage credential: {storage_credential_name}")
print(f"External location: {external_location_name}")
print(f"Security group: {security_group}")
print(f"Execute admin setup: {execute_admin_setup}")

In [ ]:
# Validación de parámetros

placeholders = []
if "<storage-account>" in raw_base_path:
    placeholders.append("raw_base_path")
if "<access-connector-resource-id>" in access_connector_resource_id:
    placeholders.append("access_connector_resource_id")

if execute_admin_setup and placeholders:
    raise ValueError(
        "No se puede ejecutar configuración administrativa con placeholders pendientes: "
        + ", ".join(placeholders)
    )

if not security_group or not security_group.strip():
    raise ValueError("security_group no puede estar vacío")

if placeholders:
    print("ADVERTENCIA: existen placeholders pendientes. El notebook mostrará los SQL, pero no los ejecutará.")

In [ ]:
# Construcción de sentencias SQL administrativas

sql_create_storage_credential = f"""
CREATE STORAGE CREDENTIAL IF NOT EXISTS {storage_credential_name}
WITH AZURE_MANAGED_IDENTITY '{access_connector_resource_id}'
"""

sql_create_external_location = f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS {external_location_name}
URL '{raw_base_path}'
WITH (STORAGE CREDENTIAL {storage_credential_name})
"""

sql_grant_external_location = f"""
GRANT READ FILES ON EXTERNAL LOCATION {external_location_name} TO `{security_group}`
"""

sql_grants_catalog = [
    f"GRANT USE CATALOG ON CATALOG {catalog_name} TO `{security_group}`",
    f"GRANT USE SCHEMA ON SCHEMA {catalog_name}.bronze TO `{security_group}`",
    f"GRANT USE SCHEMA ON SCHEMA {catalog_name}.silver TO `{security_group}`",
    f"GRANT USE SCHEMA ON SCHEMA {catalog_name}.gold TO `{security_group}`",
    f"GRANT SELECT ON SCHEMA {catalog_name}.bronze TO `{security_group}`",
    f"GRANT SELECT ON SCHEMA {catalog_name}.silver TO `{security_group}`",
    f"GRANT SELECT ON SCHEMA {catalog_name}.gold TO `{security_group}`",
]

all_sql = [
    sql_create_storage_credential,
    sql_create_external_location,
    sql_grant_external_location,
    *sql_grants_catalog,
]

print("Sentencias SQL preparadas:")
for statement in all_sql:
    print("---")
    print(statement.strip())

In [ ]:
# Ejecución controlada
# Por seguridad, solo se ejecuta si execute_admin_setup = true y no hay placeholders.

if execute_admin_setup:
    for statement in all_sql:
        spark.sql(statement)
    print("Configuración de seguridad ejecutada correctamente")
else:
    print("Modo seguro: las sentencias SQL no fueron ejecutadas.")
    print("Para ejecutarlas, reemplaza placeholders y cambia execute_admin_setup a true.")

In [ ]:
# Evidencia opcional de configuración
# Estas consultas pueden requerir permisos de metastore/admin.

try:
    print("External locations disponibles:")
    display(spark.sql("SHOW EXTERNAL LOCATIONS"))
except Exception as ex:
    print("No se pudo listar external locations con el usuario actual.")
    print(str(ex))

try:
    print("Storage credentials disponibles:")
    display(spark.sql("SHOW STORAGE CREDENTIALS"))
except Exception as ex:
    print("No se pudo listar storage credentials con el usuario actual.")
    print(str(ex))

## Resultado esperado

Cuando `execute_admin_setup = true` y los valores reales estén configurados, este notebook debe dejar listo el acceso seguro a la capa RAW mediante Managed Identity.

Cuando se mantiene `execute_admin_setup = false`, el notebook funciona como evidencia técnica/documentación ejecutable sin modificar permisos.